Task


1. Understanding BERT and XLM-RoBERTa

Objective: Learn how transformer models work and their role in NLP tasks.

Instructions:

    Read through the descriptions of BERT and XLM-RoBERTa.
    Understand how these models process text using tokenization.
    Learn about different pre-trained versions of these models and their characteristics.

Functions to use:

    from transformers import BertTokenizer, XLMRobertaTokenizer


2. Tokenizing Text

Objective: Understand how to tokenize text using pre-trained tokenizers.

Instructions:

    Use the BertTokenizer and XLMRobertaTokenizer to convert sentences into tokenized input.
    Explore the different token types, such as input_ids, attention_mask, and labels.
    Experiment with single-sentence and two-sentence tokenization.

Functions to use:

    tokenizer.encode_plus()
    tokenizer.decode()


3. Preparing Input Data for the Model

Objective: Format input data correctly for transformer models.

Instructions:

    Ensure that input sentences are padded and possibly truncated to max_length.
    Understand and set special tokens such as <s> and </s>.
    Learn about attention_mask and how it helps the model ignore padding tokens.

Functions to use:

    tokenizer.encode_plus()
    tokenizer.special_tokens_map
    tokenizer.vocab_size


4. Loading and Exploring the Dataset

Objective: Load the dataset and explore its structure.

Instructions:

    Load the training and testing data from CSV files.
    Display the first few rows to understand its structure.
    Identify the columns needed for training the model.

Functions to use:

    pd.read_csv()
    df.head()
    df.shape


5. Creating Cross-Validation Folds

Objective: Implement k-fold cross-validation for training.

Instructions:

    Use StratifiedKFold to create 5 training-validation splits.
    Ensure that each fold maintains the same label distribution.
    Store the training and validation splits in separate lists.

Functions to use:

    from sklearn.model_selection import StratifiedKFold
    kf.split()
    StratifiedKFold(shuffle=True)


In [3]:
# 1. Understanding BERT and XLM-RoBERTa

# Objective: Learn how transformer models work and their role in NLP tasks.

# Instructions:

#     Read through the descriptions of BERT and XLM-RoBERTa.
#     Understand how these models process text using tokenization.
#     Learn about different pre-trained versions of these models and their characteristics.

# Functions to use:

#     from transformers import BertTokenizer, XLMRobertaTokenizer

Both BERT and XLM-RoBERTa are Transformer-based language models that rely on the encoder architecture to understand text.
BERT understands a word's context by looking at the words to its left and right. It uses WordPiece tokenization, first splitting text based on whitespace and punctuation, then breaking words into subword units.
It is trained on two tasks: Masked Language Modeling (MLM), in which it predicts hidden words in a sentence, and Next Sentence Prediction (NSP), where it determines if one sentence logically follows another.
It has a few versions: BERT-base (12 layers, 110 million parameters);
                    BERT-large (24 layers, 340 million parameters);
                    DistilBERT (a smaller, faster version retaining 97-99% of performance while 40% smaller).

  XLM-RoBERTa (Cross-lingual RoBERTa) is a multilingual extension of RoBERTa (which is a "robustly optimized" version of BERT developed by Meta). It's designed to handle over 100 languages in one model.
  Tokenization: It uses SentencePiece tokenization with Byte-Pair Encoding (BPE). This treats the input as a raw stream of characters (including spaces), which allows it to more effectively handle languages that don't use spaces between words (e.g. Chinese, Japanese). Its vocuabulary is much larger -- around 250,000 tokens.
  It improves on BERT by removing the NSP task, training on larger datasets, and using larger batch sizes.
  Versions:
  XLM-RoBERTa-base: Standard version for cross-lingual tasks.
  XLM-RoBERTa-large: optimized for max performance.


In [4]:
from transformers import BertTokenizer, XLMRobertaTokenizer

In [5]:

# 2. Tokenizing Text

# Objective: Understand how to tokenize text using pre-trained tokenizers.

# Instructions:

#     Use the BertTokenizer and XLMRobertaTokenizer to convert sentences into tokenized input.
#     Explore the different token types, such as input_ids, attention_mask, and labels.
#     Experiment with single-sentence and two-sentence tokenization.

# Functions to use:

#     tokenizer.encode_plus()
#     tokenizer.decode()


In [6]:
from transformers import BertTokenizer, XLMRobertaTokenizer

# Initialize the tokenizers
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
xlmr_tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

sentence_1 = "This is one sample sentence."
sentence_2 = "This is another sample sentence for tokenization."

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
bert_output = bert_tokenizer(
    sentence_1,
    add_special_tokens=True,     # Adds [CLS] and [SEP]
    max_length=20,               # Set a standard length
    padding='max_length',        # Pads up to max_length
    truncation=True,             # Truncates if longer than max_length
    return_tensors='pt'          # Returns PyTorch tensors
)

print("BERT Tokens:", bert_tokenizer.convert_ids_to_tokens(bert_output['input_ids'][0]))
print("Input IDs:", bert_output['input_ids'])
print("Attention Mask:", bert_output['attention_mask'])

BERT Tokens: ['[CLS]', 'this', 'is', 'one', 'sample', 'sentence', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
Input IDs: tensor([[ 101, 2023, 2003, 2028, 7099, 6251, 1012,  102,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0]])
Attention Mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])


In [8]:
# XLM-RoBERTa Two Sentences
xlmr_output = xlmr_tokenizer(
    sentence_1,
    sentence_2,
    add_special_tokens=True,
    max_length=32,
    padding='max_length',
    truncation=True,
    return_tensors='pt'
)

# Decode to see how XLM-R separates sentences (it uses </s></s>)
decoded_text = xlmr_tokenizer.decode(xlmr_output['input_ids'][0])
print("XLMR Decoded:", decoded_text)

XLMR Decoded: <s> This is one sample sentence.</s> This is another sample sentence for tokenization.</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>


3. Preparing Input Data for the Model

Objective: Format input data correctly for transformer models.

Instructions:

    Ensure that input sentences are padded and possibly truncated to max_length.
    Understand and set special tokens such as <s> and </s>.
    Learn about attention_mask and how it helps the model ignore padding tokens.

Functions to use:

    tokenizer.encode_plus()
    tokenizer.special_tokens_map
    tokenizer.vocab_size


In [9]:
# Check the "language" your model speaks
print("BERT Specials:", bert_tokenizer.special_tokens_map)
print("XLMR Specials:", xlmr_tokenizer.special_tokens_map)

BERT Specials: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}
XLMR Specials: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}


In [10]:
sentences = ["BERT is cool.", "I am learning how to process data for RAG."]
max_len = 16

# Using the shorthand to handle padding and truncation
encoded_inputs = bert_tokenizer(
    sentences,
    padding='max_length',
    truncation=True,
    max_length=max_len,
    return_tensors='pt'
)

# Explore the vocab size
print(f"BERT Vocab Size: {bert_tokenizer.vocab_size}")

# Look at the Attention Mask for the first sentence
# Since "BERT is cool." is short, the end of this mask will be 0s
print("Attention Mask (Sentence 1):", encoded_inputs['attention_mask'][0])

# Look at the IDs to see the [PAD] tokens (usually ID 0)
print("Input IDs (Sentence 1):", encoded_inputs['input_ids'][0])

BERT Vocab Size: 30522
Attention Mask (Sentence 1): tensor([1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
Input IDs (Sentence 1): tensor([  101, 14324,  2003,  4658,  1012,   102,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0])


4. Loading and Exploring the Dataset

Objective: Load the dataset and explore its structure.

Instructions:

    Load the training and testing data from CSV files.
    Display the first few rows to understand its structure.
    Identify the columns needed for training the model.

Functions to use:

    pd.read_csv()
    df.head()
    df.shape


In [11]:
import pandas as pd
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')
df_train.shape

(12120, 6)

In [12]:
df_train.head()

,id,premise,hypothesis,lang_abv,language,label
0,5130fd2cb5,and these comments were considered in formulat...,The rules developed in the interim were put to...,en,English,0
1,5b72532a0b,These are issues that we wrestle with in pract...,Practice groups are not permitted to work on t...,en,English,2
2,3931fbe82a,Des petites choses comme celles-là font une di...,J'essayais d'accomplir quelque chose.,fr,French,0
3,5622f0c60b,you know they can't really defend themselves l...,They can't defend themselves because of their ...,en,English,0
4,86aaa48b45,ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสด...,เด็กสามารถเห็นได้ว่าชาติพันธุ์แตกต่างกันอย่างไร,th,Thai,1


In [13]:
df_test.shape

(5195, 5)

In [14]:
df_test.head()

,id,premise,hypothesis,lang_abv,language
0,c6d58c3f69,بکس، کیسی، راہیل، یسعیاہ، کیلی، کیلی، اور کولم...,"کیسی کے لئے کوئی یادگار نہیں ہوگا, کولمین ہائی...",ur,Urdu
1,cefcc82292,هذا هو ما تم نصحنا به.,عندما يتم إخبارهم بما يجب عليهم فعله ، فشلت ال...,ar,Arabic
2,e98005252c,et cela est en grande partie dû au fait que le...,Les mères se droguent.,fr,French
3,58518c10ba,与城市及其他公民及社区组织代表就IMA的艺术发展进行对话&amp,IMA与其他组织合作，因为它们都依靠共享资金。,zh,Chinese
4,c32b0d16df,Она все еще была там.,"Мы думали, что она ушла, однако, она осталась.",ru,Russian


To train a transformer model we will primarily need the following columns from train.csv:

    premise: The first sentence of the pair (Input feature).

    hypothesis: The second sentence of the pair (Input feature).

    label: The target variable (Classification output).

In [15]:
# 5. Creating Cross-Validation Folds

# Objective: Implement k-fold cross-validation for training.

# Instructions:

#     Use StratifiedKFold to create 5 training-validation splits.
#     Ensure that each fold maintains the same label distribution.
#     Store the training and validation splits in separate lists.

# Functions to use:

#     from sklearn.model_selection import StratifiedKFold
#     kf.split()
#     StratifiedKFold(shuffle=True)



In [16]:
from sklearn.model_selection import StratifiedKFold

# 1. Initialize StratifiedKFold
# n_splits=5 creates 5 distinct training-validation pairs
# shuffle=True ensures data is mixed before splitting
# random_state=42 makes sure your results are exactly the same every time you run it
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 2. Lists to store the training and validation splits
train_folds = []
val_folds = []

# 3. Create the splits using kf.split()
# We use the 'label' column to ensure each fold has the same ratio of 0, 1, and 2
for train_index, val_index in kf.split(df_train, df_train['label']):
    # Store the actual DataFrames for easy access later
    train_folds.append(df_train.iloc[train_index])
    val_folds.append(df_train.iloc[val_index])

# Check the results
print(f"Total Training Sets: {len(train_folds)}")
print(f"Total Validation Sets: {len(val_folds)}")
print(f"Sample size of Fold 1 (Validation): {len(val_folds[0])} rows")

Total Training Sets: 5
Total Validation Sets: 5
Sample size of Fold 1 (Validation): 2424 rows


In [20]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification
from tqdm.auto import tqdm

# --- 1. DATASET CLASS ---
class NLIDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128, is_test=False):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        encoding = self.tokenizer(
            str(row['premise']),
            str(row['hypothesis']),
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }
        if not self.is_test:
            item['labels'] = torch.tensor(row['label'], dtype=torch.long)
        return item

# --- 2. CONFIGURATION ---
tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- 3. SINGLE FOLD TRAINING ---
# Using only the first entry in your train_folds list
print("Starting Single Fold Training (Fold 1 of 5)...")

train_loader = DataLoader(NLIDataset(train_folds[0], tokenizer), batch_size=16, shuffle=True)
val_loader = DataLoader(NLIDataset(val_folds[0], tokenizer), batch_size=16)

model = XLMRobertaForSequenceClassification.from_pretrained('xlm-roberta-base', num_labels=3)
model.to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)

# Training for 2 epochs
for epoch in range(2):
    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for batch in pbar:
        optimizer.zero_grad()
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        targets = batch['labels'].to(device)

        outputs = model(ids, attention_mask=mask, labels=targets)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

# --- 4. SINGLE FOLD TEST PREDICTIONS ---
print("\nTraining finished. Generating final submission file...")
model.eval()
test_loader = DataLoader(NLIDataset(df_test, tokenizer, is_test=True), batch_size=16)
final_preds = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        outputs = model(ids, attention_mask=mask)
        # Directly get the class instead of saving logits since we aren't ensembling
        preds = torch.argmax(outputs.logits, dim=1)
        final_preds.extend(preds.cpu().numpy())

# --- 5. SAVE RESULTS ---
submission = pd.DataFrame({'id': df_test['id'], 'prediction': final_preds})
submission.to_csv('submission.csv', index=False)

print("\nSuccess! 'submission.csv' has been created using Fold 1.")

Starting Single Fold Training (Fold 1 of 5)...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1:   0%|          | 0/606 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/606 [00:00<?, ?it/s]


Training finished. Generating final submission file...


Predicting:   0%|          | 0/325 [00:00<?, ?it/s]


Success! 'submission.csv' has been created using Fold 1.
